## Laboratorio 2 — El tablero no cuadra
### Nombre: Karen María Mina Martínez

#### Pipeline de ventas y control de calidad

**Objetivo:**  
Construir un flujo de procesamiento que permita dejar `oro_fact_ventas` usable,
separando en `cuarentena_ventas` los registros que no cumplen las reglas del negocio.

#### Reglas principales
- Una fila por `venta_id`.
- Solo se consideran las tiendas 1, 2 y 3.
- El campo `monto` debe convertirse correctamente a número.
- Las columnas fuera del contrato no deben ingresar al hecho.
- Los registros rechazados deben conservarse en cuarentena con su respectivo motivo.
- La ejecución del notebook debe ser idempotente y no duplicar registros.

### Flujo
`Archivo de ventas → Bronce → Validación y transformación → Oro + Cuarentena`

**Herramienta:** Databricks  
**Lenguaje:** PySpark / SQL

In [0]:
# ============================================================
# 01. VERIFICACIÓN DEL ARCHIVO DE ENTRADA
# ============================================================

display(dbutils.fs.ls("file:/Workspace/"))

In [0]:
# ============================================================
# 02. UBICACIÓN DEL ARCHIVO DE VENTAS
# ============================================================
# Se revisa la carpeta de trabajo de la usuaria para localizar
# el archivo fuente antes de iniciar la ingesta.
# ============================================================

display(
    dbutils.fs.ls("file:/Workspace/Users/202400945@ujmd.edu.sv/")
)

In [0]:
# ============================================================
# 03. INGESTA DEL ARCHIVO DE VENTAS
# ============================================================
# Se carga el archivo fuente sin modificarlo y se conserva
# como vista temporal para las validaciones posteriores.
# ============================================================

ruta = "file:/Workspace/Users/202400945@ujmd.edu.sv/Laboratorio_2_KarenMina/ventas_export.csv"

bronce = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ruta)
)

bronce.createOrReplaceTempView("bronce_ventas")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

In [0]:
# ============================================================
# 04. PERFILADO INICIAL DE LAS VENTAS
# ============================================================
# Se revisan los datos originales para identificar duplicados,
# valores inválidos, formatos inconsistentes y posibles nulos
# antes de construir la capa Oro.
# ============================================================

display(bronce)

In [0]:
# ============================================================
# 05. DIAGNÓSTICO DE CONFLICTOS
# ============================================================
# Se identifican los principales problemas del archivo fuente:
# duplicados, tiendas inválidas y montos que no pueden
# convertirse correctamente a número.
# ============================================================

print("=== DUPLICADOS DE venta_id ===")

display(spark.sql("""
SELECT
    venta_id,
    COUNT(*) AS cantidad
FROM bronce_ventas
GROUP BY venta_id
HAVING COUNT(*) > 1
ORDER BY cantidad DESC, venta_id
"""))

print("=== TIENDAS EN EL ARCHIVO ===")

display(spark.sql("""
SELECT
    tienda_id,
    COUNT(*) AS cantidad
FROM bronce_ventas
GROUP BY tienda_id
ORDER BY tienda_id
"""))

print("=== MONTOS QUE NO SE PUEDEN CONVERTIR ===")

display(spark.sql("""
SELECT
    venta_id,
    monto
FROM bronce_ventas
WHERE try_cast(REPLACE(TRIM(monto), ',', '.') AS DOUBLE) IS NULL
"""))

In [0]:
# ============================================================
# 06. REVISIÓN DE REGISTROS CONFLICTIVOS
# ============================================================
# Se muestran los registros duplicados y aquellos con montos
# problemáticos para definir la regla de limpieza.
# ============================================================

print("=== REGISTROS DUPLICADOS ===")

display(spark.sql("""
SELECT *
FROM bronce_ventas
WHERE venta_id IN (
    SELECT venta_id
    FROM bronce_ventas
    GROUP BY venta_id
    HAVING COUNT(*) > 1
)
ORDER BY venta_id
"""))

print("=== REGISTROS CON MONTO PROBLEMÁTICO ===")

display(spark.sql("""
SELECT *
FROM bronce_ventas
WHERE monto IS NULL
   OR TRIM(monto) IN ('N/D', 'pendiente')
   OR TRIM(monto) = '1.250,00'
ORDER BY venta_id
"""))

In [0]:
# ============================================================
# 07. CONSTRUCCIÓN DEL PIPELINE DE VENTAS
# ============================================================
# Se aplican las reglas de calidad definidas para el laboratorio:
# - Una sola fila por venta_id.
# - Solo se aceptan las tiendas 1, 2 y 3.
# - El monto debe convertirse correctamente a número.
# - Los registros rechazados pasan a cuarentena con un motivo.
# - La columna "canal" no forma parte del hecho.
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_normalizadas AS

SELECT
    venta_id,
    fecha,
    TRY_CAST(tienda_id AS INT) AS tienda_id,
    TRY_CAST(producto_id AS INT) AS producto_id,
    TRY_CAST(cantidad AS INT) AS cantidad,

    CASE
        WHEN TRIM(monto) RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$'
            THEN TRY_CAST(
                REPLACE(
                    REPLACE(TRIM(monto), '.', ''),
                    ',', '.'
                ) AS DECIMAL(15,2)
            )
        ELSE TRY_CAST(
            REPLACE(TRIM(monto), ',', '.')
            AS DECIMAL(15,2)
        )
    END AS monto,

    medio_pago,
    canal

FROM bronce_ventas
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_clasificadas AS

WITH numeradas AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY venta_id
            ORDER BY fecha, tienda_id, producto_id, cantidad, monto
        ) AS numero_registro
    FROM ventas_normalizadas
)

SELECT
    *,
    
    CONCAT_WS(
        '; ',
        
        CASE
            WHEN numero_registro > 1
            THEN 'venta_id duplicado'
        END,
        
        CASE
            WHEN tienda_id IS NULL
              OR tienda_id NOT IN (1, 2, 3)
            THEN 'tienda no permitida'
        END,
        
        CASE
            WHEN monto IS NULL
            THEN 'monto inválido'
        END
        
    ) AS motivo

FROM numeradas
""")

# ============================================================
# 07.1. TABLA ORO
# ============================================================
# Solo ingresan los registros que cumplen las reglas de calidad.
# Se seleccionan únicamente las columnas definidas en el contrato.
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS

SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago

FROM ventas_clasificadas

WHERE numero_registro = 1
  AND tienda_id IN (1, 2, 3)
  AND monto IS NOT NULL
""")

# ============================================================
# 07.2. TABLA DE CUARENTENA
# ============================================================
# Los registros que no cumplen las reglas se conservan fuera
# del hecho y se documenta el motivo de su exclusión.
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS

SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago,
    canal,
    motivo

FROM ventas_clasificadas

WHERE motivo <> ''
""")

print("Filas en Oro:", spark.table("oro_fact_ventas").count())
print("Filas en Cuarentena:", spark.table("cuarentena_ventas").count())

In [0]:
# ============================================================
# 08. PRUEBAS DE CALIDAD DEL PIPELINE
# ============================================================
# Se validan las reglas principales del laboratorio para
# confirmar que la capa Oro y la cuarentena cumplen el contrato.
# ============================================================

oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))

print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

## Laboratorio 2 — Respuestas
### Nombre: Karen María Mina Martínez

### 1.¿Qué pasó con la venta que llegó dos veces?
Cuando una misma venta apareció más de una vez, dejamos un solo registro en la tabla Oro y enviamos los registros duplicados a cuarentena para evitar que la venta quedara repetida.

### 2. ¿Qué hicimos con el monto que venía como texto?
Convertimos los montos a formato numérico. Los valores que tenían un formato válido se transformaron correctamente, por ejemplo, 1.250,00 se convirtió en 1250.00. Los valores que no se pudieron convertir quedaron en cuarentena.

### 3. ¿Por qué dejamos fuera ciertas tiendas?
Porque la regla del laboratorio indica que solo se aceptan las tiendas 1, 2 y 3. Las ventas de las demás tiendas no cumplen con el contrato del hecho, por lo que se enviaron a cuarentena.

### 4. ¿Dónde terminó la columna que no estaba en el contrato?
La columna canal no se incluyó en la tabla Oro porque no forma parte del contrato de oro_fact_ventas. Sin embargo, se conservó en la cuarentena para no perder la información original de los registros rechazados.

### 5. Si ejecutamos nuevamente el notebook, ¿la tabla de hechos crece?
No. El flujo utiliza CREATE OR REPLACE para reconstruir las vistas, por lo que al ejecutar nuevamente el notebook no se van acumulando registros duplicados. El resultado se mantiene igual.